# PdfNet

**ID** — PDF: menggambar, menggabung, memisah, mengekstrak teks, mengenkripsi, menganotasi.
**EN** — PDF: drawing, merging, splitting, text extraction, encryption, annotation.

> Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.

Panduan lengkap / full guide: [`docs/PdfNet.md`](../docs/PdfNet.md) ·
[Bahasa Indonesia](../docs/id/PdfNet.md)

In [ ]:
// Build first:  dotnet build OfficeNet.sln -c Release

#r "../src/OfficeNet.Core/bin/Release/net10.0/Gravicode.OfficeNet.Core.dll"
#r "../src/PdfNet/bin/Release/net10.0/Gravicode.OfficeNet.PdfNet.dll"
#r "../src/WordNet/bin/Release/net10.0/Gravicode.OfficeNet.WordNet.dll"
#r "../src/OfficeNet.Rendering/bin/Release/net10.0/Gravicode.OfficeNet.Rendering.dll"

// Published instead? Swap the lines above for:
//   #r "nuget: Gravicode.OfficeNet, *"

In [ ]:
using OfficeNet.Rendering;
using Microsoft.DotNet.Interactive.Formatting;

// Renders a document and shows the first page inline, so a cell's effect is visible rather than
// described. Base64 in an <img> because the notebook has nowhere to serve a file from.
void Show(string path, int width = 520)
{
    var png = DocumentRenderer.RenderThumbnail(path, width);
    var data = Convert.ToBase64String(png);

    display(HTML($"<img src='data:image/png;base64,{data}' style='border:1px solid #ddd' />"));
}

var work = Path.Combine(Path.GetTempPath(), "officenet-notebook");
Directory.CreateDirectory(work);
string At(string name) => Path.Combine(work, name);

Console.WriteLine($"Berkas ditulis ke / files written to: {work}");

## Menggambar / Drawing

`TopDown` penting: titik asal PDF ada di kiri-bawah, sedangkan semua format Office mengukur dari
atas. /
`TopDown` matters: PDF's origin is bottom-left while every Office format measures from the top.

In [ ]:
using PdfNet.Document;
using PdfNet.Content;
using PdfNet.Security;
using OfficeNet.Core.Drawing;

var document = PdfDocument.Create();
var page = document.Pages.Add(PageSize.A4);

using (var canvas = page.OpenCanvas())
{
    canvas.TopDown = true;

    canvas.SetFillColor(OfficeColor.FromRgb(0x1F, 0x38, 0x64));
    canvas.Rectangle(0, 0, page.Width, 90).Fill();

    canvas.SetFont(StandardFont.HelveticaBold, 26);
    canvas.SetFillColor(OfficeColor.White);
    canvas.DrawText("Laporan Tahunan", 56, 56);

    canvas.SetFont(StandardFont.Helvetica, 11);
    canvas.SetFillColor(OfficeColor.Black);
    canvas.DrawText(
        "Pendapatan tumbuh 32% dibanding tahun sebelumnya, ditopang permintaan yang " +
        "kuat di Jakarta dan Bandung. Seluruh wilayah menutup tahun di atas target.",
        56, 140, page.Width - 112, TextAlignment.Justify);

    var y = 210.0;

    foreach (var (label, value) in new[] {
        ("Jakarta", 1480.0), ("Bandung", 1150.0), ("Surabaya", 905.0), ("Medan", 520.0) })
    {
        canvas.SetFillColor(OfficeColor.FromRgb(0x63, 0x8E, 0xC6));
        canvas.Rectangle(140, y, value / 4, 18).Fill();

        canvas.SetFillColor(OfficeColor.Black);
        canvas.SetFont(StandardFont.Helvetica, 10);
        canvas.DrawText(label, 56, y + 13);
        canvas.DrawText($"{value:N0}", 148 + value / 4, y + 13);

        y += 28;
    }
}

document.Info.Title = "Laporan Tahunan";
document.Info.Author = "Gravicode Studios";
document.Save(At("gambar.pdf"));

Show(At("gambar.pdf"));

## Ekstraksi teks / Text extraction

PDF tidak memuat teks dalam urutan baca — ia memuat instruksi menggambar. Mengekstrak berarti
memainkan ulang operatornya dan *menyimpulkan spasi yang tidak pernah disimpan*. /
A PDF does not contain text in reading order — it contains drawing instructions. Extracting means
replaying the operators and *inferring the spaces the file never stored*.

In [ ]:
using var reopened = PdfDocument.Open(At("gambar.pdf"));

Console.WriteLine(reopened.ExtractText());
Console.WriteLine(new string('-', 60));

foreach (var fragment in reopened.Pages[0].ExtractTextFragments().Take(6))
    Console.WriteLine($"{fragment.Text,-28} @ ({fragment.X,6:0.#}, {fragment.Y,6:0.#})  {fragment.FontSize}pt");

## Menggabung dan memisah / Merge and split

Penggabungan menulis ulang setiap nomor objek, jadi dua berkas yang sama-sama menyebut font-nya
`7 0 R` tidak bertabrakan. /
Merging rewrites every object number, so two files that both call their font `7 0 R` do not collide.

In [ ]:
using var wordDocument = WordNet.WordDocument.Create();
wordDocument.AddHeading("Lampiran", 1);
wordDocument.AddParagraph("Halaman tambahan dari WordNet.");
wordDocument.SaveAsPdf(At("lampiran.pdf"));

using var a = PdfDocument.Open(At("gambar.pdf"));
using var b = PdfDocument.Open(At("lampiran.pdf"));

Console.WriteLine($"Sebelum: {a.Pages.Count} halaman");
a.Merge(b);
Console.WriteLine($"Sesudah: {a.Pages.Count} halaman");

a.Save(At("gabungan.pdf"));

var parts = PdfDocument.Open(At("gabungan.pdf")).Split();
Console.WriteLine($"Dipisah menjadi {parts.Count} dokumen");
foreach (var part in parts) part.Dispose();

## Anotasi dan watermark / Annotations and watermarks

Titik quad sorotan berurutan kiri-atas, kanan-atas, kiri-bawah, kanan-bawah. Searah jarum jam
menghasilkan bentuk dasi kupu-kupu. /
Highlight quad points go upper-left, upper-right, lower-left, lower-right. Clockwise draws a bowtie.

In [ ]:
using PdfNet.Annotations;

using var annotated = PdfDocument.Open(At("gambar.pdf"));
var target = annotated.Pages[0];

target.AddWatermark("DRAF", OfficeColor.Gray, 0.10);
target.AddTextNote(500, 200, "Perlu ditinjau sebelum rilis.", "Kang Fadhil");
target.AddHighlight([new PdfRectangle(56, 130, 300, 150)], OfficeColor.FromRgb(0xFF, 0xF0, 0x00));

annotated.Save(At("anotasi.pdf"));

Console.WriteLine($"{target.GetAnnotations().Count} anotasi");
Show(At("anotasi.pdf"));

## Enkripsi / Encryption

In [ ]:
using (var secret = PdfDocument.Open(At("gambar.pdf")))
{
    secret.Encrypt(
        userPassword: "buka",
        ownerPassword: "pemilik",
        permissions: PdfPermissions.Print,
        cipher: PdfCipher.Aes256);

    secret.Save(At("terkunci.pdf"));
}

try
{
    using var _ = PdfDocument.Open(At("terkunci.pdf"));
    Console.WriteLine("Terbuka tanpa kata sandi — tidak seharusnya!");
}
catch (Exception ex)
{
    Console.WriteLine($"Tanpa kata sandi: {ex.GetType().Name}");
}

using var unlocked = PdfDocument.Open(At("terkunci.pdf"), "buka");
Console.WriteLine($"Dengan kata sandi: {unlocked.Pages.Count} halaman, izin = {unlocked.Permissions}");

In [ ]:
document.Dispose();